# Objective-Only MVP: OUTCOME_LEADS From Raw Evidence To Recommendation

This notebook intentionally uses **Meta objective only**. It does not use any
additional campaign taxonomy to choose a KPI, form a peer group, calculate a
benchmark, make a decision, allocate illustrative budget, or instruct the LLM.

The worked example is **Post-Eid Lookalike Test**, and its peers are the other
campaigns whose objective is `OUTCOME_LEADS`.

The notebook keeps every important number visible:

1. raw records and eligible records;
2. numerator, denominator, and raw rate;
3. peer evidence before and after the half-count adjustment;
4. Empirical-Bayes prior and corrected score;
5. uncertainty ranges and the range-based decision;
6. campaign, ad set, ad, creative, and audience scorecards;
7. conversation semantics as diagnostic evidence;
8. a deterministic 70/30 scenario and optional structured LLM narration.

## 0. End-To-End Logic

```text
Sample 2 JSON files
      |
      v
Canonical media + WhatsApp outcome tables
      |
      v
Select one objective and its KPI contract
      |
      v
Aggregate campaign / ad set / ad / creative / audience
      |
      v
Raw rate + same-objective peer evidence
      |
      v
Empirical-Bayes corrected score + uncertainty range
      |
      v
SCALE / HOLD / KILL from the lift range
      |
      +--> conversation semantics explain possible reasons
      |
      v
70 exploit / 30 named exploration tests
      |
      v
Optional LLM explanation of locked evidence
```

**Statistical engine owns:** calculations, ranges, decisions, and budget units.

**LLM owns:** interpretation in plain language, hypotheses, and narration. It is
not allowed to change the locked values.

## 1. Load The Real Pipeline And Data

The notebook uses the production normalizer and aggregations. The statistical
comparison is defined locally so its objective-only behavior is easy to audit.
Optional LLM calls are off by default.

In [1]:
import hashlib
import json
import os
from math import sqrt
from pathlib import Path
from typing import List, Optional

import altair as alt
import pandas as pd
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display
from pydantic import BaseModel

from src_2.analytics import build_scorecards
from src_2.analytics.empirical_bayes import fit_beta_prior, score_beta_binomial
from src_2.application import load_conversation_signal_records
from src_2.ingestion import build_data_quality_report, load_sample2, normalize_cycle
from src_2.intelligence import build_semantic_input, merge_token_usage

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")
alt.data_transformers.disable_max_rows()

ROOT = Path.cwd()
if not (ROOT / "src_2").exists():
    raise RuntimeError("Run this notebook from the Team2-MarketingExpert repository root.")

load_dotenv(ROOT / ".env")
INPUT = ROOT / "src_2" / "data" / "input" / "sampe_2"
configured_signal_path = os.getenv("CONVERSATION_SIGNALS_PATH", "").strip()
SIGNALS = (
    Path(configured_signal_path).expanduser()
    if configured_signal_path
    else ROOT / "src_2" / "artifacts" / "conversation_signals_v2_paid.jsonl"
)
if not SIGNALS.is_absolute():
    SIGNALS = ROOT / SIGNALS
PILOT_SIGNALS = (
    ROOT / "src_2" / "artifacts" / "conversation_signals_v3_paid.jsonl"
)
PILOT_SIZE = 10
LLM_CACHE = ROOT / "outputs" / "objective_leads_learning_llm"
FOCUS_OBJECTIVE = "OUTCOME_LEADS"
POST_EID_NAME = "Post-Eid Lookalike Test"

## 2. See The Objectives Before Choosing A KPI

An objective is the outcome Meta was asked to optimize delivery toward. For this
MVP, it is also the only grouping field used to define comparable campaigns.

This avoids intersecting two classification systems. A campaign enters the
`OUTCOME_LEADS` analysis because its objective is `OUTCOME_LEADS`, and for no
other reason.

In [2]:
raw_payload = load_sample2(INPUT)
canonical = normalize_cycle(raw_payload)
quality = build_data_quality_report(canonical)
signal_records = load_conversation_signal_records(SIGNALS) if SIGNALS.exists() else []

objective_meanings = {
    "OUTCOME_AWARENESS": "Reach people and create visibility; business outcomes are supporting evidence.",
    "OUTCOME_ENGAGEMENT": "Generate interaction with the ad or content.",
    "OUTCOME_LEADS": "Generate leads or conversations that can progress toward an order.",
    "OUTCOME_SALES": "Generate purchase-related outcomes and delivered customer value.",
}

objective_map = canonical.campaigns.groupby("objective", as_index=False).agg(
    campaign_count=("campaign_name", "nunique"),
    campaigns=("campaign_name", lambda values: "\n".join(sorted(values))),
)
objective_map["business meaning"] = objective_map["objective"].map(objective_meanings)
objective_map = objective_map[["objective", "business meaning", "campaign_count", "campaigns"]]
display(objective_map.style.set_properties(
    subset=["campaigns"], **{"white-space": "pre-wrap", "text-align": "left"}
))

,objective,business meaning,campaign_count,campaigns
0,OUTCOME_AWARENESS,Reach people and create visibility; business outcomes are supporting evidence.,2,Awareness Boost January Summer Retention Push
1,OUTCOME_ENGAGEMENT,Generate interaction with the ad or content.,1,January Trial Bundle Test
2,OUTCOME_LEADS,Generate leads or conversations that can progress toward an order.,3,Lookalike Scale Cycle 3 Post-Eid Lookalike Test Pre-Ramadan Bundle Promo
3,OUTCOME_SALES,Generate purchase-related outcomes and delivered customer value.,6,Always-On Premium Acquisition Eid Gifting Premium Mid-Year Sale Ramadan Iftar Premium Bundles Ramadan Suhoor Specials Summer Premium Launch


## 2A. Schema-v3 Paid Pilot: What Are These 10 Records?

Before using semantic signals in scorecard explanations, inspect the output itself.
This chapter always selects the **first 10 validated records** from the versioned
schema-v3 paid artifact, even after the artifact grows to 617 conversations.

The pilot was outcome-stratified to exercise different conversation shapes. It is
useful for schema validation and qualitative review, but it is **not a representative
sample for estimating portfolio rates**.

Each conversation went through two independent model calls:

1. **Semantic extraction:** redacted message text became structured intent, need,
   barriers, products, commercial signals, stage, summary, and agent evaluation.
2. **Ad-message alignment:** the creative promise was compared with the validated
   conversation need.

Neither call received customer identity, phone, address, order outcome, revenue, campaign
score, or budget. Those deterministic fields are joined only afterward for analysis.

In [3]:
all_v3_records = (
    load_conversation_signal_records(PILOT_SIGNALS)
    if PILOT_SIGNALS.exists()
    else []
)
pilot_signal_records = all_v3_records[:PILOT_SIZE]
if len(pilot_signal_records) != PILOT_SIZE:
    raise RuntimeError(
        f"Expected at least {PILOT_SIZE} schema-v3 pilot records at {PILOT_SIGNALS}; "
        f"found {len(pilot_signal_records)}."
    )

conversation_lookup = canonical.conversations.set_index("conversation_id")
campaign_lookup = canonical.campaigns.set_index("campaign_id")
adset_names = canonical.adsets.set_index("adset_id")["adset_name"].to_dict()
ad_names = canonical.ads.set_index("ad_id")["ad_name"].to_dict()
creative_names = (
    canonical.creatives.set_index("creative_id")["creative_name"].to_dict()
)

pilot_rows = []
for record in pilot_signal_records:
    outcome = conversation_lookup.loc[record.conversation_id]
    campaign = campaign_lookup.loc[record.attribution.campaign_id]
    signals = record.signals
    total_usage = merge_token_usage(
        record.semantic_usage, record.ad_match_usage
    )
    pilot_rows.append({
        "conversation_id": record.conversation_id,
        "objective": campaign["objective"],
        "campaign": campaign["campaign_name"],
        "adset": adset_names.get(record.attribution.adset_id),
        "ad": ad_names.get(record.attribution.ad_id),
        "creative": creative_names.get(record.attribution.creative_id),
        "audience": record.attribution.audience_type,
        "actual_outcome_after_join": outcome["outcome_type"],
        "mature_outcome": bool(outcome["is_mature_outcome"]),
        "order_observed": bool(outcome["has_order"]),
        "purpose": signals.conversation_purpose.value,
        "customer_need": signals.customer_need,
        "purchase_intent": signals.purchase_intent.level.value,
        "stage": signals.conversation_stage.value,
        "urgency": signals.urgency.level.value,
        "price_sensitivity": signals.price_sensitivity.level.value,
        "deal_seeking": signals.deal_seeking.level.value,
        "delivery_intent": signals.delivery_intent.level.value,
        "sales_agreement": signals.sales_agreement.level.value,
        "specificity": signals.specificity.level.value,
        "financing": signals.financing.level.value,
        "competitor_mentioned": signals.competitor_mention.mentioned,
        "stated_exit_reason": signals.stated_exit_reason.reason.value,
        "next_step_agreed": signals.next_step_agreed.agreed,
        "barrier_count": len(signals.barriers),
        "product_count": len(signals.mentioned_products),
        "value_driver_count": len(signals.value_drivers),
        "commercial_trait_count": len(signals.commercial_traits),
        "agent_helpfulness": signals.agent_evaluation.helpfulness.value,
        "agent_progression": signals.agent_evaluation.progression.value,
        "agent_tone_quality": signals.agent_tone.quality.value,
        "ad_message_match": record.ad_message_match.level.value,
        "schema_version": record.signal_schema_version,
        "prompt_version": record.prompt_version,
        "model": record.model,
        "usage_logged": total_usage.request_count > 0,
        "api_output": signals.conversation_summary,
    })

pilot = pd.DataFrame(pilot_rows)
pilot_snapshot = pd.DataFrame([
    ["Validated records inspected", len(pilot)],
    ["Objectives represented", pilot["objective"].nunique()],
    ["Campaigns represented", pilot["campaign"].nunique()],
    ["Schema versions", ", ".join(map(str, sorted(pilot["schema_version"].unique())))],
    ["Prompt versions", ", ".join(sorted(pilot["prompt_version"].unique()))],
    ["Records with API usage logging", int(pilot["usage_logged"].sum())],
], columns=["pilot fact", "value"])
display(pilot_snapshot)

,pilot fact,value
0,Validated records inspected,10
1,Objectives represented,3
2,Campaigns represented,7
3,Schema versions,3
4,Prompt versions,conversation-signals-v3.2
5,Records with API usage logging,0


## 2B. Understand The Nested Output Contract

A JSONL line is one `ConversationSignalRecord`. It has five conceptual blocks:

| Block | What it contains | Why it exists |
|---|---|---|
| Identity and attribution | Conversation, campaign, ad set, ad, creative, audience | Joins semantics back to the correct marketing entity |
| Version and provenance | Input, prompt, schema, model, timestamp | Makes every label reproducible and auditable |
| Conversation signals | Intent, need, barriers, products, stage, commercial traits, agent quality | Explains what the conversation appears to contain |
| Ad-message match | Alignment level, rationale, evidence indexes | Tests whether the acquired need matches the creative promise |
| API usage | Tokens, cached tokens, reasoning tokens, price snapshot, cost | Measures extraction cost and includes validation retries |

Nested objects preserve meaning. For example, `purchase_intent` is not only a
label: it also says whether the agent elicited it and which message indexes support it.
Arrays such as `barriers` preserve multiple observations instead of forcing one label.

In [4]:
example_record = pilot_signal_records[0]
example_payload = example_record.model_dump(mode="json")

def flatten_structure(value, path="record"):
    rows = []
    if isinstance(value, dict):
        if not value:
            rows.append({"path": path, "data type": "object", "example": "{}"})
        for key, child in value.items():
            rows.extend(flatten_structure(child, f"{path}.{key}"))
    elif isinstance(value, list):
        rows.append({
            "path": path,
            "data type": "list",
            "example": f"{len(value)} item(s)",
        })
        if value:
            rows.extend(flatten_structure(value[0], f"{path}[]"))
    else:
        text = "null" if value is None else str(value)
        rows.append({
            "path": path,
            "data type": type(value).__name__,
            "example": text[:100],
        })
    return rows

structure_table = pd.DataFrame(flatten_structure(example_payload))
display(structure_table.style.set_properties(
    subset=["path", "example"], **{"text-align": "left"}
))
print("Expandable example JSON for", example_record.conversation_id)
display(JSON(example_payload, expanded=False))

,path,data type,example
0,record.conversation_id,str,conv_068
1,record.attribution.source_platform,str,meta_ctwa
2,record.attribution.campaign_id,str,120209876543220008
3,record.attribution.adset_id,str,120209876543240016
4,record.attribution.ad_id,str,120209876543210025
5,record.attribution.creative_id,str,120209876543230012
6,record.attribution.audience_type,str,lookalike
7,record.input_projection_version,str,semantic-input-v3
8,record.prompt_version,str,conversation-signals-v3.2
9,record.prompt_sha256,str,5baf356732fc82879dfa93082a030ad1e79618ec68559dd4f438331db6973d4c


Expandable example JSON for conv_068


<IPython.core.display.JSON object>

## 2C. Provenance And Attribution For All 10

Attribution is deterministic. The LLM does not choose the campaign, ad set, ad,
creative, or audience. These identifiers come from the canonical source join.

The outcome column is deliberately labeled **after join**: it is useful for later
diagnostics but was never part of either model request.

In [5]:
provenance_columns = [
    "conversation_id", "objective", "campaign", "adset", "ad", "creative",
    "audience", "actual_outcome_after_join", "schema_version",
    "prompt_version", "model",
]
display(pilot[provenance_columns].style.set_properties(
    subset=["campaign", "adset", "ad", "creative"],
    **{"text-align": "left"},
))

,conversation_id,objective,campaign,adset,ad,creative,audience,actual_outcome_after_join,schema_version,prompt_version,model
0,conv_068,OUTCOME_LEADS,Post-Eid Lookalike Test,10% Lookalike experimental,10% LAL test creative,Post-Eid Lookalike test creative B,lookalike,active,3,conversation-signals-v3.2,gpt-5-mini
1,conv_016,OUTCOME_LEADS,Lookalike Scale Cycle 3,1% Cycle 2 high-LTV scale,Lookalike 1% scale all-cycle3,Lookalike scale premium quality,lookalike,adversarial,3,conversation-signals-v3.2,gpt-5-mini
2,conv_010,OUTCOME_AWARENESS,Awareness Boost January,Cairo+Alex broad awareness,Awareness creative Cairo+Alex,Always-on premium acquisition default,broad,cancelled,3,conversation-signals-v3.2,gpt-5-mini
3,conv_003,OUTCOME_AWARENESS,Summer Retention Push,Existing customer Custom Audience retention,Retention Custom Audience warm,Summer retention awareness warm,custom,delivered,3,conversation-signals-v3.2,gpt-5-mini
4,conv_006,OUTCOME_SALES,Always-On Premium Acquisition,1% Lookalike all-delivered customers,1% LAL acquisition initial,Lookalike scale premium quality,lookalike,ghosted,3,conversation-signals-v3.2,gpt-5-mini
5,conv_009,OUTCOME_SALES,Mid-Year Sale,General audience sale promo,Sale urgency refresh,Mid-year sale urgency,broad,refunded,3,conversation-signals-v3.2,gpt-5-mini
6,conv_001,OUTCOME_SALES,Ramadan Iftar Premium Bundles,1% Lookalike premium customers Ramadan,Iftar 1% LAL creative A,Iftar premium bundle,lookalike,stuck_pending,3,conversation-signals-v3.2,gpt-5-mini
7,conv_123,OUTCOME_SALES,Always-On Premium Acquisition,1% Lookalike all-delivered customers,1% LAL acquisition refresh,Always-on premium acquisition default,lookalike,active,3,conversation-signals-v3.2,gpt-5-mini
8,conv_135,OUTCOME_SALES,Ramadan Iftar Premium Bundles,1% Lookalike premium customers Ramadan,Iftar 1% LAL creative B refresh scarcity,Iftar bundle scarcity,lookalike,adversarial,3,conversation-signals-v3.2,gpt-5-mini
9,conv_011,OUTCOME_SALES,Always-On Premium Acquisition,Cairo broad acquisition,Cairo broad creative A initial,Always-on premium acquisition default,broad,cancelled,3,conversation-signals-v3.2,gpt-5-mini


## 2D. Flatten The Semantic Interpretation

This view turns nested JSON into one row per conversation for exploration. It does
not replace the nested artifact. `unknown` and `not_assessable` are valid evidence
states, not failures and not zeros.

Read each row as: “Given only the redacted messages, the model classified this
conversation this way, with evidence indexes available for verification.”

In [6]:
core_columns = [
    "conversation_id", "purpose", "customer_need", "purchase_intent", "stage",
    "barrier_count", "product_count", "agent_helpfulness", "agent_progression",
    "agent_tone_quality", "ad_message_match",
]
commercial_columns = [
    "conversation_id", "urgency", "price_sensitivity", "deal_seeking",
    "delivery_intent", "sales_agreement", "specificity", "financing",
    "competitor_mentioned", "stated_exit_reason", "next_step_agreed",
    "value_driver_count", "commercial_trait_count",
]

display(Markdown("### Core semantic fields"))
display(pilot[core_columns].style.set_properties(
    subset=["customer_need"], **{"text-align": "left"}
))
display(Markdown("### Added commercial diagnostic fields"))
display(pilot[commercial_columns])

### Core semantic fields

,conversation_id,purpose,customer_need,purchase_intent,stage,barrier_count,product_count,agent_helpfulness,agent_progression,agent_tone_quality,ad_message_match
0,conv_068,product_information,معلومات تفصيلية وقصص المنتج (مثل مصدر العسل وطريقة تحميص البن) لاختيار هدايا,low,discovery,0,2,good,strong,positive,aligned
1,conv_016,product_information,العميل يسأل عن سعر المنتج.,low,discovery,0,0,not_assessable,not_assessable,not_assessable,partial
2,conv_010,purchase,شراء Madagascar vanilla beans و sage tea,high,checkout,1,2,strong,strong,positive,mismatch
3,conv_003,purchase,"Place an order for groceries: 1L olive oil, sourdough, and parmesan if available.",high,checkout,1,3,good,strong,positive,partial
4,conv_006,product_information,looking for genuine saffron,low,discovery,1,1,good,adequate,neutral,mismatch
5,conv_009,return_or_refund,"Place an order (3 Truffles, 1 Saffron, 2 Coffee Lover Boxes) and later return one sealed Coffee Lover Box",high,post_purchase,1,3,strong,strong,positive,aligned
6,conv_001,purchase,طلب Iftar Bundle للعزومة غداً مع توصيل قبل المغرب,high,checkout,1,1,good,good,positive,aligned
7,conv_123,purchase,"Order 6 office gift baskets, ~500 EGP per person",medium,consideration,0,1,good,adequate,positive,aligned
8,conv_135,promotion_information,مَعرفة السعر أو تفاصيل العرض,low,discovery,0,0,not_assessable,not_assessable,not_assessable,partial
9,conv_011,purchase,"Purchase a trial bundle (light honey, dark roast coffee, sweets).",high,checkout,1,1,strong,strong,positive,aligned


### Added commercial diagnostic fields

,conversation_id,urgency,price_sensitivity,deal_seeking,delivery_intent,sales_agreement,specificity,financing,competitor_mentioned,stated_exit_reason,next_step_agreed,value_driver_count,commercial_trait_count
0,conv_068,none,unknown,none,none,none,medium,none,False,not_stated,True,2,1
1,conv_016,none,interest,none,none,none,none,none,False,not_stated,False,1,0
2,conv_010,none,unknown,none,checkout_details,complete,medium,none,False,changed_mind,True,0,0
3,conv_003,none,unknown,none,checkout_details,complete,high,none,False,not_stated,True,0,1
4,conv_006,none,sensitive,none,none,none,low,none,False,price,False,1,1
5,conv_009,high,none,none,checkout_details,complete,high,none,False,changed_mind,True,0,1
6,conv_001,high,none,none,checkout_details,partial,high,none,False,not_stated,True,0,0
7,conv_123,low,interest,none,none,none,high,none,False,not_stated,False,1,0
8,conv_135,none,interest,interested,none,none,none,none,False,not_stated,False,1,0
9,conv_011,none,unknown,unknown,readiness,complete,high,none,False,unknown,True,1,1


## 2E. Known Versus Unknown Evidence

Coverage answers a different question from performance:

- **Known:** the conversation contained enough evidence to assign a value.
- **Unknown/not assessable:** the evidence was absent or insufficient.

A signal with low known coverage should not be turned into a campaign percentage
without displaying its assessable denominator.

In [7]:
coverage_fields = {
    "Purpose": "purpose",
    "Customer need": "customer_need",
    "Purchase intent": "purchase_intent",
    "Stage": "stage",
    "Urgency": "urgency",
    "Price sensitivity": "price_sensitivity",
    "Deal seeking": "deal_seeking",
    "Delivery intent": "delivery_intent",
    "Sales agreement": "sales_agreement",
    "Specificity": "specificity",
    "Financing": "financing",
    "Agent helpfulness": "agent_helpfulness",
    "Agent progression": "agent_progression",
    "Agent tone": "agent_tone_quality",
    "Ad-message match": "ad_message_match",
}
unknown_values = {"unknown", "not_assessable", "", None}
coverage_rows = []
for label, column in coverage_fields.items():
    for value in pilot[column]:
        status = (
            "Unknown / not assessable"
            if pd.isna(value) or value in unknown_values
            else "Known"
        )
        coverage_rows.append({"signal": label, "status": status})
coverage = (
    pd.DataFrame(coverage_rows)
    .groupby(["signal", "status"], as_index=False)
    .size()
    .rename(columns={"size": "conversations"})
)
coverage["share"] = coverage["conversations"] / len(pilot)

coverage_chart = (
    alt.Chart(coverage)
    .mark_bar()
    .encode(
        y=alt.Y(
            "signal:N",
            sort=list(coverage_fields),
            title=None,
        ),
        x=alt.X(
            "conversations:Q",
            stack="normalize",
            axis=alt.Axis(format="%"),
            title="Share of the 10 pilot conversations",
        ),
        color=alt.Color(
            "status:N",
            scale=alt.Scale(
                domain=["Known", "Unknown / not assessable"],
                range=["#187a6b", "#d6d9dc"],
            ),
            title=None,
        ),
        tooltip=[
            alt.Tooltip("signal:N"),
            alt.Tooltip("status:N"),
            alt.Tooltip("conversations:Q"),
            alt.Tooltip("share:Q", format=".0%"),
        ],
    )
    .properties(height=420, title="Semantic assessability in the schema-v3 pilot")
)
display(coverage_chart)

alt.Chart(...)

## 2F. Explore Repeated Arrays Without Losing Detail

Barriers, products, value drivers, and commercial traits are one-to-many fields.
Exploding them creates analytical child tables while keeping the original
conversation ID as the join key.

- Barrier severity describes how strongly an objection blocks progress.
- Barrier resolution requires later customer evidence, not only an agent answer.
- Canonical product ID is nullable when a reference cannot be matched safely.
- Value drivers capture what matters to the customer.
- Commercial traits capture reusable demand patterns such as feature priority.

In [8]:
barrier_rows, product_rows, driver_rows, trait_rows = [], [], [], []
for record in pilot_signal_records:
    conversation_id = record.conversation_id
    for item in record.signals.barriers:
        barrier_rows.append({
            "conversation_id": conversation_id,
            "barrier_type": item.barrier_type.value,
            "severity": item.severity.value,
            "resolution": item.resolution.value,
            "description": item.description,
            "evidence_indexes": item.evidence_message_indexes,
        })
    for item in record.signals.mentioned_products:
        product_rows.append({
            "conversation_id": conversation_id,
            "product_reference": item.product_reference,
            "canonical_product_id": item.canonical_product_id,
            "evidence_indexes": item.evidence_message_indexes,
        })
    for item in record.signals.value_drivers:
        driver_rows.append({
            "conversation_id": conversation_id,
            "value_driver": item.driver.value,
            "evidence_indexes": item.evidence_message_indexes,
        })
    for item in record.signals.commercial_traits:
        trait_rows.append({
            "conversation_id": conversation_id,
            "trait": item.trait.value,
            "strength": item.strength.value,
            "detail": item.detail,
            "evidence_indexes": item.evidence_message_indexes,
        })

def show_child_table(title, rows):
    display(Markdown(f"### {title}"))
    frame = pd.DataFrame(rows)
    if frame.empty:
        print("No records in this pilot.")
    else:
        display(frame.style.set_properties(**{"text-align": "left"}))

show_child_table("Barriers", barrier_rows)
show_child_table("Mentioned products", product_rows)
show_child_table("Value drivers", driver_rows)
show_child_table("Commercial traits", trait_rows)

### Barriers

,conversation_id,barrier_type,severity,resolution,description,evidence_indexes
0,conv_010,product_fit,blocking,unresolved,العميل طلب إلغاء لأن لديهم الكثير من vanilla extract بالفعل,[5]
1,conv_003,product_availability,minor,resolved,Customer requested parmesan but agent stated store does not carry it.,"[2, 3, 4]"
2,conv_006,price,moderate,unresolved,customer says the price is expensive,[2]
3,conv_009,product_fit,minor,unresolved,Customer over-ordered and requests to return one sealed Coffee Lover Box,[4]
4,conv_001,agent_experience,blocking,unresolved,العميل ضغط على 'عرض' لكنه لم يتلقَ رد/تأكيد بعد، مما أوقف تقدم الطلب,[5]
5,conv_011,product_fit,blocking,unresolved,Customer requests cancellation because a household member began a health regimen and advised stopping sugar.,[5]


### Mentioned products

,conversation_id,product_reference,canonical_product_id,evidence_indexes
0,conv_068,عسل,None,[2]
1,conv_068,قهوة / بن,None,[2]
2,conv_010,Madagascar vanilla beans,None,[0]
3,conv_010,sage tea,None,[0]
4,conv_003,olive oil 1L,prod_063,[2]
5,conv_003,sourdough,prod_091,[2]
6,conv_003,parmesan,None,[2]
7,conv_006,saffron,None,[0]
8,conv_009,Truffles (3),None,[0]
9,conv_009,Saffron (1),None,[0]


### Value drivers

,conversation_id,value_driver,evidence_indexes
0,conv_068,gifting,[2]
1,conv_068,product_fit,[2]
2,conv_016,price,"[0, 1, 2, 3]"
3,conv_006,price,[2]
4,conv_123,price,[2]
5,conv_135,price,"[0, 1, 2, 3]"
6,conv_011,health,[5]


### Commercial traits

,conversation_id,trait,strength,detail,evidence_indexes
0,conv_068,feature_priority,high,تعطي أولوية لقصص المنتج ومصدره عند اختيار هدايا,[2]
1,conv_003,feature_priority,medium,specified 1L size for olive oil,[2]
2,conv_006,feature_priority,medium,seeking authentic/high-quality saffron,[0]
3,conv_009,bulk_purchase_interest,medium,ordered multiple units of products,[0]
4,conv_011,feature_priority,high,"Specified light honey, dark roast, and sweets",[0]


## 2G. Trace One Output Back To Redacted Evidence

Evidence indexes are positions in the exact redacted message list sent to the
semantic extractor. They allow a reviewer to verify a label without storing raw
transcript text in the signal artifact.

The example below reconstructs the privacy-safe input locally. The model saw role,
relative timing, redacted text, and product references. It did not see customer or
outcome objects.

In [9]:
example_id = pilot_signal_records[0].conversation_id
raw_by_id = {str(item["id"]): item for item in raw_payload.conversations}
example_input = build_semantic_input(raw_by_id[example_id])
example_signals = pilot_signal_records[0].signals

redacted_messages = pd.DataFrame([
    {
        "message_index": message.message_index,
        "role": message.role.value,
        "relative_minute": message.relative_minute,
        "redacted_text": message.redacted_text,
        "product_references": message.product_references,
    }
    for message in example_input.messages
])
display(Markdown(f"### Redacted model input: {example_id}"))
display(redacted_messages.style.set_properties(
    subset=["redacted_text"], **{"text-align": "left", "white-space": "pre-wrap"}
))

evidence_rows = []
def add_evidence(signal, value, indexes):
    evidence_rows.append({
        "signal": signal,
        "extracted_value": value,
        "evidence_message_indexes": indexes,
    })

add_evidence(
    "purchase_intent",
    example_signals.purchase_intent.level.value,
    example_signals.purchase_intent.evidence_message_indexes,
)
add_evidence(
    "urgency",
    example_signals.urgency.level.value,
    example_signals.urgency.evidence_message_indexes,
)
add_evidence(
    "delivery_intent",
    example_signals.delivery_intent.level.value,
    example_signals.delivery_intent.evidence_message_indexes,
)
add_evidence(
    "sales_agreement",
    example_signals.sales_agreement.level.value,
    example_signals.sales_agreement.evidence_message_indexes,
)
add_evidence(
    "next_step_agreed",
    example_signals.next_step_agreed.agreed,
    example_signals.next_step_agreed.evidence_message_indexes,
)
for position, item in enumerate(example_signals.barriers, start=1):
    add_evidence(
        f"barrier_{position}:{item.barrier_type.value}",
        f"{item.severity.value} / {item.resolution.value}",
        item.evidence_message_indexes,
    )
for position, item in enumerate(example_signals.mentioned_products, start=1):
    add_evidence(
        f"product_{position}",
        item.product_reference,
        item.evidence_message_indexes,
    )
add_evidence(
    "agent_evaluation",
    example_signals.agent_evaluation.helpfulness.value,
    example_signals.agent_evaluation.evidence_message_indexes,
)
add_evidence(
    "ad_message_match",
    pilot_signal_records[0].ad_message_match.level.value,
    pilot_signal_records[0].ad_message_match.evidence_message_indexes,
)
display(Markdown("### Output-to-evidence map"))
display(pd.DataFrame(evidence_rows))

### Redacted model input: conv_068

,message_index,role,relative_minute,redacted_text,product_references
0,0,customer,0.000000,أهلاً. عاوزة احكي مع حد، عندي أسئلة كتير عن المنتجات,[]
1,1,agent,3.000000,أهلاً [REDACTED_GREETING_NAME]. أيوة، نقدر نساعدك. أنهي categories بتجذبك؟,[]
2,2,customer,6.000000,أنا بحب أهدي. اللي بحب فيه إن المنتج يكون له story. زي العسل من فين، البن ازاي يتحمص,[]
3,3,agent,10.000000,بنفهم تماماً. كل منتج عندنا له story. ابعتي رقمك ولا الإيميل وهنبعتلك dossier فيه قصص كل منتج. لو حابة نتكلم تليفون متاحين كمان.,[]
4,4,customer,12.000000,ابعتي على الإيميل [REDACTED_EMAIL]. هاخد وقتي اقرا,[]


### Output-to-evidence map

,signal,extracted_value,evidence_message_indexes
0,purchase_intent,low,"[0, 2, 4]"
1,urgency,none,[4]
2,delivery_intent,none,[]
3,sales_agreement,none,[]
4,next_step_agreed,True,[4]
5,product_1,عسل,[2]
6,product_2,قهوة / بن,[2]
7,agent_evaluation,good,"[1, 3]"
8,ad_message_match,aligned,[2]


## 2H. Join Outcomes Only After Extraction

This is the governed handoff:

`privacy-safe semantic output + deterministic attribution + structured outcome`

The joined table can help investigate hypotheses such as whether unresolved price
barriers appear more often in cancelled conversations. With only 10 deliberately
stratified records, these are examples of questions, not reliable findings.

The first pilot predates API usage logging. Therefore zero logged requests means
**historical usage unavailable**, not zero cost. New records will contain actual
semantic and ad-alignment usage separately.

In [10]:
outcome_diagnostic = pilot[[
    "conversation_id", "objective", "campaign", "actual_outcome_after_join",
    "mature_outcome", "order_observed", "purchase_intent", "stage",
    "price_sensitivity", "barrier_count", "stated_exit_reason",
    "ad_message_match",
]]
display(outcome_diagnostic)

usage_rows = []
for record in pilot_signal_records:
    for stage_name, usage in [
        ("semantic extraction", record.semantic_usage),
        ("ad-message alignment", record.ad_match_usage),
    ]:
        usage_rows.append({
            "conversation_id": record.conversation_id,
            "stage": stage_name,
            "requests": usage.request_count,
            "input_tokens": usage.input_tokens,
            "cached_input_tokens": usage.cached_input_tokens,
            "output_tokens": usage.output_tokens,
            "reasoning_output_tokens": usage.reasoning_output_tokens,
            "estimated_cost_usd": usage.estimated_cost_usd,
            "status": (
                "logged" if usage.request_count else "historical usage unavailable"
            ),
        })
usage_detail = pd.DataFrame(usage_rows)
display(Markdown("### Usage logging by LLM stage"))
display(usage_detail)

,conversation_id,objective,campaign,actual_outcome_after_join,mature_outcome,order_observed,purchase_intent,stage,price_sensitivity,barrier_count,stated_exit_reason,ad_message_match
0,conv_068,OUTCOME_LEADS,Post-Eid Lookalike Test,active,False,False,low,discovery,unknown,0,not_stated,aligned
1,conv_016,OUTCOME_LEADS,Lookalike Scale Cycle 3,adversarial,True,False,low,discovery,interest,0,not_stated,partial
2,conv_010,OUTCOME_AWARENESS,Awareness Boost January,cancelled,True,True,high,checkout,unknown,1,changed_mind,mismatch
3,conv_003,OUTCOME_AWARENESS,Summer Retention Push,delivered,True,True,high,checkout,unknown,1,not_stated,partial
4,conv_006,OUTCOME_SALES,Always-On Premium Acquisition,ghosted,True,False,low,discovery,sensitive,1,price,mismatch
5,conv_009,OUTCOME_SALES,Mid-Year Sale,refunded,True,True,high,post_purchase,none,1,changed_mind,aligned
6,conv_001,OUTCOME_SALES,Ramadan Iftar Premium Bundles,stuck_pending,False,True,high,checkout,none,1,not_stated,aligned
7,conv_123,OUTCOME_SALES,Always-On Premium Acquisition,active,False,False,medium,consideration,interest,0,not_stated,aligned
8,conv_135,OUTCOME_SALES,Ramadan Iftar Premium Bundles,adversarial,True,False,low,discovery,interest,0,not_stated,partial
9,conv_011,OUTCOME_SALES,Always-On Premium Acquisition,cancelled,True,True,high,checkout,unknown,1,unknown,aligned


### Usage logging by LLM stage

,conversation_id,stage,requests,input_tokens,cached_input_tokens,output_tokens,reasoning_output_tokens,estimated_cost_usd,status
0,conv_068,semantic extraction,0,0,0,0,0,None,historical usage unavailable
1,conv_068,ad-message alignment,0,0,0,0,0,None,historical usage unavailable
2,conv_016,semantic extraction,0,0,0,0,0,None,historical usage unavailable
3,conv_016,ad-message alignment,0,0,0,0,0,None,historical usage unavailable
4,conv_010,semantic extraction,0,0,0,0,0,None,historical usage unavailable
5,conv_010,ad-message alignment,0,0,0,0,0,None,historical usage unavailable
6,conv_003,semantic extraction,0,0,0,0,0,None,historical usage unavailable
7,conv_003,ad-message alignment,0,0,0,0,0,None,historical usage unavailable
8,conv_006,semantic extraction,0,0,0,0,0,None,historical usage unavailable
9,conv_006,ad-message alignment,0,0,0,0,0,None,historical usage unavailable


## 2I. How To Read This Pilot Correctly

1. Validate structure, privacy, evidence indexes, and attribution first.
2. Read `unknown` as missing evidence, never as a negative customer signal.
3. Keep arrays at child-table grain before aggregating them.
4. Join outcomes after extraction, then use semantics to explain possible reasons.
5. Do not use this outcome-stratified group of 10 to estimate campaign prevalence.
6. Do not allow semantic labels to change the Empirical-Bayes funding score.
7. After all 617 records pass audit, aggregate counts and assessable denominators at
   campaign, ad set, ad, creative, and audience levels.

## 3. The Objective-To-KPI Contract

The lookup below is an explicit MVP assumption. The objective chooses one primary
statistical score; the other metrics stay visible as supporting evidence and
guardrails. No weighted composite score is created.

| Objective | Primary statistical score | Why |
|---|---|---|
| Awareness | Link responses / impressions | Available media response proxy; not proof of awareness lift |
| Engagement | Link responses / impressions | Available interaction proxy in this sample |
| Leads | Mature conversations with an order / mature conversations | Uses WhatsApp progression after unresolved cases are removed |
| Sales | Delivered unique customers / mature unique customers | Uses completed customer-level delivery outcome |

This notebook calculates only the `OUTCOME_LEADS` row.

In [11]:
OBJECTIVE_KPI_LOOKUP = {
    "OUTCOME_AWARENESS": {
        "metric": "link_ctr",
        "numerator": "link_clicks",
        "denominator": "impressions",
        "business_job": "Create visibility and measurable response.",
        "success_question": "Did the campaign create more link response than comparable awareness campaigns?",
    },
    "OUTCOME_ENGAGEMENT": {
        "metric": "link_ctr",
        "numerator": "link_clicks",
        "denominator": "impressions",
        "business_job": "Generate interaction with the ad or content.",
        "success_question": "Did the campaign create more link response than comparable engagement campaigns?",
    },
    "OUTCOME_LEADS": {
        "metric": "order_creation_rate",
        "numerator": "mature_orders_created",
        "denominator": "mature_conversations",
        "business_job": "Generate WhatsApp leads that progress to a created order.",
        "success_question": "Did mature WhatsApp conversations progress to orders better than comparable lead campaigns?",
    },
    "OUTCOME_SALES": {
        "metric": "customer_delivered_rate",
        "numerator": "delivered_customers",
        "denominator": "mature_unique_customers",
        "business_job": "Generate delivered customer outcomes.",
        "success_question": "Did mature unique customers reach delivery better than comparable sales campaigns?",
    },
}

objective_spec = OBJECTIVE_KPI_LOOKUP[FOCUS_OBJECTIVE]
display(pd.DataFrame([
    ["Objective", FOCUS_OBJECTIVE],
    ["Business job", objective_spec["business_job"]],
    ["Success question", objective_spec["success_question"]],
    ["Primary score", objective_spec["metric"]],
    ["Numerator", objective_spec["numerator"]],
    ["Denominator", objective_spec["denominator"]],
    ["Better direction", "higher"],
], columns=["contract element", "value"]))

,contract element,value
0,Objective,OUTCOME_LEADS
1,Business job,Generate WhatsApp leads that progress to a created order.
2,Success question,Did mature WhatsApp conversations progress to orders better than comparable lead campaigns?
3,Primary score,order_creation_rate
4,Numerator,mature_orders_created
5,Denominator,mature_conversations
6,Better direction,higher


## 4. Build Raw Aggregates At Five Levels

The aggregation layer keeps media facts and observed WhatsApp outcomes at their
correct grains. It does not multiply daily media rows by conversation rows.

The score uses `mature_orders_created / mature_conversations`. Spend, delivered
revenue, return on ad spend, cancellation, refund, and media response remain
supporting evidence. They are not mixed into the score with arbitrary weights.

In [12]:
raw_scorecards = build_scorecards(canonical, signal_records)
levels = ["campaign", "adset", "ad", "creative", "audience"]

inventory_rows = []
for level in levels:
    frame = raw_scorecards.by_level(level)
    focused = frame.loc[frame["objective"].eq(FOCUS_OBJECTIVE)]
    inventory_rows.append({
        "level": level,
        "all entities": len(frame),
        "same-objective entities": len(focused),
        "mature conversations": int(focused["mature_conversations"].sum()),
        "mature orders": int(focused["mature_orders_created"].sum()),
    })
display(pd.DataFrame(inventory_rows))

,level,all entities,same-objective entities,mature conversations,mature orders
0,campaign,12,3,129,96
1,adset,26,6,129,96
2,ad,40,8,129,96
3,creative,31,7,129,96
4,audience,22,4,129,96


## 5. Define The Objective-Only Empirical-Bayes Scorer

For each entity:

1. keep entities at the same hierarchy level;
2. keep only `OUTCOME_LEADS`;
3. remove the entity being scored from its peer evidence;
4. require at least two valid peers;
5. fit a Beta prior from peer successes and trials;
6. update it with the entity's evidence;
7. compare posterior draws with peer benchmark draws;
8. decide from the 95% favorable-lift range.

This is partial pooling: a small sample leans more toward its objective peer
benchmark, while a large sample is driven more by its own evidence.

In [13]:
def stable_seed(level, entity_id, metric):
    raw = f"{level}|{entity_id}|{metric}".encode("utf-8")
    return int(hashlib.sha256(raw).hexdigest()[:8], 16)


def score_objective_level(frame, level, objective, spec):
    scoped = frame.loc[frame["objective"].eq(objective)].copy()
    results = []
    for _, row in scoped.iterrows():
        peers = scoped.loc[scoped["entity_id"].ne(row["entity_id"])].copy()
        peers["_successes"] = pd.to_numeric(peers[spec["numerator"]], errors="coerce")
        peers["_trials"] = pd.to_numeric(peers[spec["denominator"]], errors="coerce")
        peers = peers.loc[
            peers["_trials"].gt(0)
            & peers["_successes"].ge(0)
            & peers["_successes"].le(peers["_trials"])
        ]
        if len(peers) < 2:
            results.append({
                "entity_id": row["entity_id"],
                "raw_score": row[spec["numerator"]] / row[spec["denominator"]]
                if row[spec["denominator"]] else None,
                "corrected_score": None,
                "corrected_score_low": None,
                "corrected_score_high": None,
                "benchmark_score": None,
                "expected_lift": None,
                "lift_low": None,
                "lift_high": None,
                "probability_better": None,
                "decision": "insufficient_evidence",
                "benchmark_peer_count": len(peers),
                "benchmark_source": "insufficient same-objective peers",
                "prior_alpha": None,
                "prior_beta": None,
                "prior_strength": None,
            })
            continue

        prior = fit_beta_prior(
            peers["_successes"], peers["_trials"],
            source="current-cycle same-objective empirical prior",
        )
        score = score_beta_binomial(
            metric=spec["metric"],
            numerator=spec["numerator"],
            denominator=spec["denominator"],
            direction="higher",
            successes=float(row[spec["numerator"]]),
            trials=float(row[spec["denominator"]]),
            prior=prior,
            practical_lift_threshold=0.0,
            seed=stable_seed(level, row["entity_id"], spec["metric"]),
            samples=5000,
        )
        payload = score.model_dump(mode="json")
        payload["decision"] = score.decision.value
        payload["entity_id"] = row["entity_id"]
        results.append(payload)
    return scoped.merge(pd.DataFrame(results), on="entity_id", how="left")


objective_scorecards = {
    level: score_objective_level(
        raw_scorecards.by_level(level), level, FOCUS_OBJECTIVE, objective_spec
    )
    for level in levels
}

## 6. Campaign-Level Result For All Lead Campaigns

Raw score answers: *what fraction progressed in this observed mature sample?*

Corrected score answers: *after accounting for sample size and peer evidence,
what underlying rate is most plausible?*

Decision answers: *does the full favorable-lift range support better, worse, or
uncertain performance?*

In [14]:
campaign_scores = objective_scorecards["campaign"].copy()
campaign_view_columns = [
    "campaign_id", "campaign_name", "observed_conversations",
    "open_or_pending_conversations", "mature_orders_created",
    "mature_conversations", "raw_score", "corrected_score",
    "corrected_score_low", "corrected_score_high", "benchmark_score",
    "lift_low", "lift_high", "probability_better", "decision",
    "spend", "net_revenue", "net_roas",
]
campaign_view = campaign_scores[campaign_view_columns].sort_values(
    "corrected_score", ascending=False
)
display(campaign_view.style.format({
    "raw_score": "{:.2%}", "corrected_score": "{:.2%}",
    "corrected_score_low": "{:.2%}", "corrected_score_high": "{:.2%}",
    "benchmark_score": "{:.2%}", "lift_low": "{:+.2%}",
    "lift_high": "{:+.2%}", "probability_better": "{:.2%}",
    "spend": "{:,.2f}", "net_revenue": "{:,.2f}", "net_roas": "{:.2f}",
}))

,campaign_id,campaign_name,observed_conversations,open_or_pending_conversations,mature_orders_created,mature_conversations,raw_score,corrected_score,corrected_score_low,corrected_score_high,benchmark_score,lift_low,lift_high,probability_better,decision,spend,net_revenue,net_roas
2,120209876543220010,Lookalike Scale Cycle 3,36,2,27,34,79.41%,75.32%,65.49%,84.04%,72.40%,-12.31%,+19.02%,64.00%,hold,"31,465.87","37,936.00",1.21
1,120209876543220008,Post-Eid Lookalike Test,63,11,38,52,73.08%,73.90%,64.66%,82.35%,75.00%,-16.22%,+16.19%,44.04%,hold,"7,762.12","42,145.00",5.43
0,120209876543220003,Pre-Ramadan Bundle Promo,46,3,31,43,72.09%,73.69%,64.07%,82.17%,75.29%,-16.87%,+14.62%,40.56%,hold,"13,958.72","28,905.00",2.07


## 7. Visualize Corrected Campaign Scores And Their Ranges

A point alone can make small differences look decisive. The horizontal line is
the corrected-score range. Overlap means the ranking is uncertain, even if one
point is numerically larger.

In [15]:
plot_frame = campaign_view.copy()
base = alt.Chart(plot_frame).encode(
    y=alt.Y("campaign_name:N", sort="-x", title=None),
    tooltip=[
        alt.Tooltip("campaign_name:N", title="Campaign"),
        alt.Tooltip("mature_orders_created:Q", title="Orders"),
        alt.Tooltip("mature_conversations:Q", title="Mature conversations"),
        alt.Tooltip("raw_score:Q", format=".2%", title="Raw"),
        alt.Tooltip("corrected_score:Q", format=".2%", title="Corrected"),
        alt.Tooltip("decision:N", title="Decision"),
    ],
)
ranges = base.mark_rule(strokeWidth=4, color="#68737d").encode(
    x=alt.X("corrected_score_low:Q", axis=alt.Axis(format="%"), title="Order creation rate"),
    x2="corrected_score_high:Q",
)
points = base.mark_point(filled=True, size=120, color="#176b87").encode(
    x="corrected_score:Q"
)
display((ranges + points).properties(height=190, title="Lead campaigns: corrected score with 95% range"))

alt.LayerChart(...)

# Worked Example: Post-Eid Lookalike Test

The next sections reconstruct the Post-Eid result from raw rows. Nothing is hidden
behind the final percentage.

## 8. Raw Outcome Ledger

`active` and `stuck_pending` are unresolved. They are not counted as failures.
The score denominator includes only mature conversations.

In [16]:
post_row = campaign_scores.loc[campaign_scores["campaign_name"].eq(POST_EID_NAME)].iloc[0]
post_id = str(post_row["campaign_id"])
post_conversations = canonical.conversations.loc[
    canonical.conversations["campaign_id"].eq(post_id)
].copy()

outcome_ledger = post_conversations.groupby("outcome_type", as_index=False).agg(
    conversations=("conversation_id", "nunique"),
    mature=("is_mature_outcome", "sum"),
    created_orders=("has_mature_order", "sum"),
    delivered=("is_delivered", "sum"),
    cancelled=("is_cancelled", "sum"),
    refunded=("is_refunded", "sum"),
)
display(outcome_ledger)

observed = int(len(post_conversations))
unresolved = int(post_conversations["is_open_or_pending"].sum())
mature = int(post_conversations["is_mature_outcome"].sum())
successes = int(post_conversations["has_mature_order"].sum())
print(f"Observed = {observed}")
print(f"Unresolved = {unresolved}")
print(f"Mature denominator = {observed} - {unresolved} = {mature}")
print(f"Created-order numerator = {successes}")

,outcome_type,conversations,mature,created_orders,delivered,cancelled,refunded
0,active,7,0,0,0,0,0
1,adversarial,2,2,0,0,0,0
2,cancelled,7,7,7,0,7,0
3,delivered,27,27,27,27,0,0
4,ghosted,12,12,0,0,0,0
5,refunded,4,4,4,0,0,4
6,stuck_pending,4,0,0,0,0,0


Observed = 63
Unresolved = 11
Mature denominator = 63 - 11 = 52
Created-order numerator = 38


## 9. Raw Rate And Its Raw Confidence Interval

The raw rate is descriptive:

```text
raw rate = created-order conversations / mature conversations
         = 38 / 52
```

The Wilson interval shows how uncertain that raw sample proportion is without
using peer information. It is preferred to the simple normal approximation for
binary rates, especially with small samples or rates near 0% or 100%.

In [17]:
def wilson_interval(success_count, trial_count, z=1.959963984540054):
    if trial_count <= 0:
        return None, None
    rate = success_count / trial_count
    denominator = 1 + z * z / trial_count
    center = (rate + z * z / (2 * trial_count)) / denominator
    margin = z * sqrt(
        rate * (1 - rate) / trial_count + z * z / (4 * trial_count * trial_count)
    ) / denominator
    return center - margin, center + margin

raw_rate = successes / mature
raw_low, raw_high = wilson_interval(successes, mature)
print(f"Raw rate = {successes} / {mature} = {raw_rate:.4%}")
print(f"Raw 95% Wilson interval = [{raw_low:.4%}, {raw_high:.4%}]")
assert abs(raw_rate - post_row["raw_score"]) < 1e-12

Raw rate = 38 / 52 = 73.0769%
Raw 95% Wilson interval = [59.7478%, 83.2310%]


## 10. Build The Benchmark From Same-Objective Peers

The campaign is removed from its own benchmark. The two remaining lead campaigns
provide the comparison evidence. Their raw rates stay visible so we can see where
the benchmark came from.

In [18]:
peer_rows = campaign_scores.loc[campaign_scores["campaign_id"].ne(post_id)].copy()
peer_rows = peer_rows[[
    "campaign_name", "mature_orders_created", "mature_conversations", "order_creation_rate"
]]
display(peer_rows.style.format({"order_creation_rate": "{:.2%}"}))

peer_successes = int(peer_rows["mature_orders_created"].sum())
peer_trials = int(peer_rows["mature_conversations"].sum())
pooled_peer_rate = peer_successes / peer_trials
print(f"Pooled raw peer rate = {peer_successes} / {peer_trials} = {pooled_peer_rate:.4%}")

,campaign_name,mature_orders_created,mature_conversations,order_creation_rate
0,Pre-Ramadan Bundle Promo,31,43,72.09%
2,Lookalike Scale Cycle 3,27,34,79.41%


Pooled raw peer rate = 58 / 77 = 75.3247%


## 11. Why Add 0.5 And 1?

The benchmark uses a half-success continuity adjustment:

```text
adjusted peer rate = (peer successes + 0.5) / (peer trials + 1)
                   = (58 + 0.5) / (77 + 1)
                   = 75.00%
```

The `+1` is one tiny synthetic observation split into `0.5` success and `0.5`
failure. It prevents an all-success or all-failure peer sample from claiming the
true rate is exactly 100% or 0%. It is a numerical stabilization convention, not
a real customer and not a hidden business target. With 77 peer observations its
effect is small: 75.32% becomes 75.00%.

In [19]:
adjusted_peer_rate = (peer_successes + 0.5) / (peer_trials + 1.0)
adjustment_effect = adjusted_peer_rate - pooled_peer_rate
print(f"Before adjustment = {pooled_peer_rate:.4%}")
print(f"After adjustment  = ({peer_successes} + 0.5) / ({peer_trials} + 1) = {adjusted_peer_rate:.4%}")
print(f"Effect             = {adjustment_effect:+.4%}")
assert abs(adjusted_peer_rate - post_row["benchmark_score"]) < 1e-12

Before adjustment = 75.3247%
After adjustment  = (58 + 0.5) / (77 + 1) = 75.0000%
Effect             = -0.3247%


## 12. Convert The Benchmark Into An Empirical Prior

A prior needs two things:

- **center:** the typical peer rate, 75.00%;
- **strength:** how many equivalent observations the peer information should
  contribute.

We do not simply inject all 77 peer conversations into every campaign. That would
make the benchmark too powerful and pretend the campaigns are identical.

The strength calculation is deliberately conservative:

1. average peer size = `77 / 2 = 38.5`;
2. estimate a strength implied by how much the two peer rates differ;
3. use the smaller of those values, with a minimum of 1.

If peers disagree strongly, the variance-implied strength becomes smaller and the
prior has less influence. Here the average peer size, 38.5, is the limiting value.

In [20]:
fitted_prior = fit_beta_prior(
    peer_rows["mature_orders_created"],
    peer_rows["mature_conversations"],
    source="current-cycle same-objective empirical prior",
)
prior_table = pd.DataFrame([
    ["Prior center", fitted_prior.mean, "Typical adjusted peer rate"],
    ["Prior strength", fitted_prior.strength, "Equivalent evidence allowed from peers"],
    ["Prior alpha", fitted_prior.alpha, "Prior success-shaped evidence"],
    ["Prior beta", fitted_prior.beta, "Prior failure-shaped evidence"],
    ["Peer count", fitted_prior.peer_count, "Independent peer campaign aggregates"],
], columns=["component", "value", "business interpretation"])
display(prior_table)
print(f"alpha = center x strength = {fitted_prior.mean:.6f} x {fitted_prior.strength:.4f} = {fitted_prior.alpha:.4f}")
print(f"beta  = (1 - center) x strength = {(1-fitted_prior.mean):.6f} x {fitted_prior.strength:.4f} = {fitted_prior.beta:.4f}")

,component,value,business interpretation
0,Prior center,0.750000,Typical adjusted peer rate
1,Prior strength,38.500000,Equivalent evidence allowed from peers
2,Prior alpha,28.875000,Prior success-shaped evidence
3,Prior beta,9.625000,Prior failure-shaped evidence
4,Peer count,2.000000,Independent peer campaign aggregates


alpha = center x strength = 0.750000 x 38.5000 = 28.8750
beta  = (1 - center) x strength = 0.250000 x 38.5000 = 9.6250


## 13. Update The Prior With Post-Eid Evidence

A Beta prior is convenient because we can update it by adding observed successes
and failures:

```text
posterior alpha = prior alpha + observed successes
posterior beta  = prior beta + observed failures
corrected score = posterior alpha / (posterior alpha + posterior beta)
```

The corrected score is a transparent weighted average of the 75.00% peer center
and the 73.08% campaign raw rate. Post-Eid contributes 52 observations; the prior
contributes 38.5 equivalent observations, so Post-Eid still has more influence.

In [21]:
failures = mature - successes
posterior_alpha = fitted_prior.alpha + successes
posterior_beta = fitted_prior.beta + failures
corrected_score = posterior_alpha / (posterior_alpha + posterior_beta)
prior_weight = fitted_prior.strength / (fitted_prior.strength + mature)
data_weight = mature / (fitted_prior.strength + mature)

calculation = pd.DataFrame([
    ["Observed successes", successes],
    ["Observed failures", failures],
    ["Prior alpha", fitted_prior.alpha],
    ["Prior beta", fitted_prior.beta],
    ["Posterior alpha", posterior_alpha],
    ["Posterior beta", posterior_beta],
    ["Prior weight", prior_weight],
    ["Campaign-data weight", data_weight],
    ["Corrected score", corrected_score],
], columns=["quantity", "value"])
display(calculation)
print(f"Corrected score = {posterior_alpha:.3f} / ({posterior_alpha:.3f} + {posterior_beta:.3f}) = {corrected_score:.4%}")
assert abs(corrected_score - post_row["corrected_score"]) < 1e-12

,quantity,value
0,Observed successes,38.000000
1,Observed failures,14.000000
2,Prior alpha,28.875000
3,Prior beta,9.625000
4,Posterior alpha,66.875000
5,Posterior beta,23.625000
6,Prior weight,0.425414
7,Campaign-data weight,0.574586
8,Corrected score,0.738950


Corrected score = 66.875 / (66.875 + 23.625) = 73.8950%


## 14. From A Corrected Point To A Decision Range

The engine simulates plausible campaign rates from the posterior and plausible
benchmark rates from the prior. For a higher-is-better metric:

```text
favorable lift = campaign rate - benchmark rate
```

- `SCALE`: the lower 95% lift bound is above zero.
- `KILL`: the upper 95% lift bound is below zero.
- `HOLD`: the range crosses zero.

Post-Eid's range crosses zero. The data supports both a modestly worse and a
modestly better underlying result, so the honest decision is HOLD.

In [22]:
post_decision_table = pd.DataFrame([
    ["Raw score", post_row["raw_score"]],
    ["Corrected score", post_row["corrected_score"]],
    ["Corrected low", post_row["corrected_score_low"]],
    ["Corrected high", post_row["corrected_score_high"]],
    ["Benchmark", post_row["benchmark_score"]],
    ["Expected favorable lift", post_row["expected_lift"]],
    ["Lift low", post_row["lift_low"]],
    ["Lift high", post_row["lift_high"]],
    ["Probability better", post_row["probability_better"]],
], columns=["result", "value"])
display(post_decision_table.style.format({"value": "{:.2%}"}))
print(f"Decision = {post_row['decision'].upper()} because the lift range "
      f"[{post_row['lift_low']:+.2%}, {post_row['lift_high']:+.2%}] crosses zero.")
assert post_row["decision"] == "hold"

,result,value
0,Raw score,73.08%
1,Corrected score,73.90%
2,Corrected low,64.66%
3,Corrected high,82.35%
4,Benchmark,75.00%
5,Expected favorable lift,-0.85%
6,Lift low,-16.22%
7,Lift high,16.19%
8,Probability better,44.04%


Decision = HOLD because the lift range [-16.22%, +16.19%] crosses zero.


## 15. Post-Eid Scorecards At Every Level

The same objective KPI and same-objective peer rule are used at every level. This
allows a campaign manager to inspect ad sets, ads, creatives, and audiences
without trusting a small raw rate.

Important reading rule: a numerically larger corrected score is a **leader to
test**, not automatically a winner. The range-based decision remains authoritative.

In [23]:
detail_rows = []
for level, frame in objective_scorecards.items():
    selected = frame.loc[frame["campaign_id"].eq(post_id)].copy()
    for _, row in selected.iterrows():
        detail_rows.append({
            "level": level,
            "entity_id": row["entity_id"],
            "entity": row["entity_name"],
            "orders": row["mature_orders_created"],
            "mature conversations": row["mature_conversations"],
            "raw": row["raw_score"],
            "corrected": row["corrected_score"],
            "low": row["corrected_score_low"],
            "high": row["corrected_score_high"],
            "benchmark": row["benchmark_score"],
            "lift low": row["lift_low"],
            "lift high": row["lift_high"],
            "probability better": row["probability_better"],
            "decision": row["decision"],
            "spend": row["spend"],
            "net revenue": row["net_revenue"],
        })
post_detail = pd.DataFrame(detail_rows)
display(post_detail.style.format({
    "raw": "{:.2%}", "corrected": "{:.2%}", "low": "{:.2%}",
    "high": "{:.2%}", "benchmark": "{:.2%}",
    "lift low": "{:+.2%}", "lift high": "{:+.2%}",
    "probability better": "{:.2%}", "spend": "{:,.2f}",
    "net revenue": "{:,.2f}",
}))

,level,entity_id,entity,orders,mature conversations,raw,corrected,low,high,benchmark,lift low,lift high,probability better,decision,spend,net revenue
0,campaign,120209876543220008,Post-Eid Lookalike Test,38,52,73.08%,73.90%,64.66%,82.35%,75.00%,-16.22%,+16.19%,44.04%,hold,"7,762.12","42,145.00"
1,adset,120209876543240015,5% Lookalike experimental,18,22,81.82%,78.94%,63.51%,90.95%,72.69%,-20.80%,+39.08%,64.20%,hold,"4,256.10","16,259.00"
2,adset,120209876543240016,10% Lookalike experimental,20,30,66.67%,69.37%,55.10%,82.32%,76.50%,-30.40%,+24.83%,28.52%,hold,"3,506.02","25,886.00"
3,ad,120209876543210024,5% LAL test creative,18,22,81.82%,80.19%,63.96%,92.65%,72.69%,-24.94%,+50.92%,59.32%,hold,"4,256.10","16,259.00"
4,ad,120209876543210025,10% LAL test creative,20,30,66.67%,68.07%,52.19%,82.37%,76.50%,-37.51%,+34.25%,28.78%,hold,"3,506.02","25,886.00"
5,creative,120209876543220008::creative::120209876543230011,Post-Eid Lookalike test creative A,18,22,81.82%,80.11%,63.68%,92.59%,72.69%,-23.72%,+50.07%,60.12%,hold,"4,256.10","16,259.00"
6,creative,120209876543220008::creative::120209876543230012,Post-Eid Lookalike test creative B,20,30,66.67%,68.12%,51.88%,82.24%,76.50%,-37.88%,+34.28%,30.42%,hold,"3,506.02","25,886.00"
7,audience,120209876543220008::audience::lookalike,Lookalike,38,52,73.08%,73.71%,63.43%,83.16%,75.00%,-19.84%,+19.38%,42.64%,hold,"7,762.12","42,145.00"


## 16. Compare Post-Eid Children Visually

Post-Eid has two ad sets, two ads, and two creatives, but only one audience label
after audience aggregation. We can compare the two configured lookalike widths at
ad-set level; the audience-level label alone cannot distinguish 5% from 10%.

In [24]:
child_plot = post_detail.loc[post_detail["level"].isin(["adset", "ad", "creative"])].copy()
base = alt.Chart(child_plot).encode(
    y=alt.Y("entity:N", sort="-x", title=None),
    color=alt.Color("level:N", title="Level", scale=alt.Scale(
        domain=["adset", "ad", "creative"], range=["#176b87", "#c45a2a", "#5a6f3b"]
    )),
    tooltip=[
        alt.Tooltip("level:N"), alt.Tooltip("entity:N"),
        alt.Tooltip("orders:Q"), alt.Tooltip("mature conversations:Q"),
        alt.Tooltip("raw:Q", format=".2%"),
        alt.Tooltip("corrected:Q", format=".2%"),
        alt.Tooltip("decision:N"),
    ],
)
child_ranges = base.mark_rule(strokeWidth=3).encode(
    x=alt.X("low:Q", axis=alt.Axis(format="%"), title="Corrected order creation rate"),
    x2="high:Q",
)
child_points = base.mark_point(filled=True, size=90).encode(x="corrected:Q")
display((child_ranges + child_points).properties(height=310, title="Post-Eid child entities"))

alt.LayerChart(...)

## 17. What Conversation Semantics Contributes

Structured outcomes tell us **what happened**: order, delivery, cancellation,
refund, unresolved, and revenue. Conversation semantics can suggest **why**:

- customer goal and purchase intent;
- barriers and whether the agent helped;
- product demand;
- urgency, price sensitivity, deals, agreement, and next steps;
- alignment between the ad promise and the customer need.

These are diagnostic signals, not funding-score ingredients. Otherwise subjective
LLM labels could silently override real business outcomes.

In [25]:
post_semantics = objective_scorecards["campaign"].loc[
    objective_scorecards["campaign"]["campaign_id"].eq(post_id)
].iloc[0]
schema_versions = pd.Series(
    [record.signal_schema_version for record in signal_records], dtype="Int64"
).value_counts().sort_index()

semantic_summary = pd.DataFrame([
    ["Semantic records", post_semantics["semantic_conversations"]],
    ["High purchase intent", post_semantics["high_purchase_intent_conversations"]],
    ["Conversation has a barrier", post_semantics["barrier_conversations"]],
    ["Agent rated helpful", post_semantics["agent_helpful_conversations"]],
    ["Top purpose", post_semantics["top_conversation_purpose"]],
    ["Top barrier", post_semantics["top_barrier"]],
    ["Top mentioned product", post_semantics["top_mentioned_product"]],
], columns=["signal", "raw evidence"])
display(semantic_summary)
print("Signal schema versions present:")
display(schema_versions.rename("records").to_frame())
print("The current 63 records use schema version 1. Newer high-value fields are "
      "present in the contract but were not extracted, so unknown must not be read as absence.")

,signal,raw evidence
0,Semantic records,63.000000
1,High purchase intent,41.000000
2,Conversation has a barrier,30.000000
3,Agent rated helpful,59.000000
4,Top purpose,purchase
5,Top barrier,product_fit
6,Top mentioned product,Hostess Premium


Signal schema versions present:


,records
1,63


The current 63 records use schema version 1. Newer high-value fields are present in the contract but were not extracted, so unknown must not be read as absence.


## 18. Semantic Evidence By Post-Eid Ad Set

The percentages are accompanied by counts. This avoids treating 19/25 and 22/38
as equally precise. The signals generate hypotheses; they do not establish that
an ad set caused the customer's intent or barrier.

In [26]:
post_adsets = objective_scorecards["adset"].loc[
    objective_scorecards["adset"]["campaign_id"].eq(post_id)
].copy()
semantic_adset_view = post_adsets[[
    "entity_name", "semantic_conversations", "high_purchase_intent_conversations",
    "high_purchase_intent_rate", "barrier_conversations", "barrier_conversation_rate",
    "agent_helpful_conversations", "agent_helpful_rate", "top_barrier",
    "top_mentioned_product", "mature_orders_created", "mature_conversations",
    "raw_score", "corrected_score", "decision",
]]
display(semantic_adset_view.style.format({
    "high_purchase_intent_rate": "{:.2%}", "barrier_conversation_rate": "{:.2%}",
    "agent_helpful_rate": "{:.2%}", "raw_score": "{:.2%}",
    "corrected_score": "{:.2%}",
}))

,entity_name,semantic_conversations,high_purchase_intent_conversations,high_purchase_intent_rate,barrier_conversations,barrier_conversation_rate,agent_helpful_conversations,agent_helpful_rate,top_barrier,top_mentioned_product,mature_orders_created,mature_conversations,raw_score,corrected_score,decision
2,5% Lookalike experimental,25.000000,19.000000,79.17%,14.000000,56.00%,23.000000,100.00%,product_fit,Eid Deluxe,18,22,81.82%,78.94%,hold
3,10% Lookalike experimental,38.000000,22.000000,59.46%,16.000000,42.11%,36.000000,94.74%,price,Coffee Lover,20,30,66.67%,69.37%,hold


## 19. High-Value Semantic Fields Still To Extract

Schema version 1 gives useful intent, barriers, products, and agent-evaluation
evidence. The following fields can sharpen the next test once they are actually
extracted. They must remain **unknown**, not zero, until then.

| Signal | Business use |
|---|---|
| Urgency | Detect time-sensitive demand |
| Price sensitivity and deal seeking | Separate normal price questions from price-driven abandonment |
| Delivery intent and sales agreement | Detect checkout readiness and agreement |
| Barrier severity and resolution | Separate minor questions from unresolved blockers |
| Stated exit reason | Explain cancellation, ghosting, or abandonment |
| Competitor mention and value driver | Understand alternatives and what customers value |
| Ad-message alignment | Test whether the conversation need matches the ad promise |
| Next-step order progression | Test whether an agreement was followed by an observed order |

Once available, aggregate each as `count / assessable conversations`, and always
report unknown coverage beside it.

The order-progression field is a proxy. It does not prove that every promised
follow-up action was completed.

## 20. Data Quality Boundary

The observed WhatsApp records are the business-outcome evidence, but they reconcile
to only a small fraction of Meta-attributed conversation starts. Therefore this
notebook can rank the observed supplied sample, but the budget scenario is not yet
operational for the full population.

In [27]:
quality_view = pd.DataFrame([
    ["Meta-attributed conversation starts", quality.meta_conversation_starts],
    ["Observed Meta-sourced WhatsApp conversations", quality.observed_meta_whatsapp_conversations],
    ["Observed / Meta reconciliation ratio", quality.reconciliation_ratio],
    ["Organic/direct conversations kept as context", quality.organic_direct_conversations],
    ["Open or pending outcomes", quality.open_or_pending_conversations],
    ["Repeated customers", quality.repeated_customers],
    ["Reach greater than impressions rows", quality.reach_exceeds_impressions_rows],
    ["Evidence status", quality.status.value],
], columns=["quality fact", "value"])
display(quality_view)
print(f"Coverage = {quality.observed_meta_whatsapp_conversations:,} / "
      f"{quality.meta_conversation_starts:,} = {quality.reconciliation_ratio:.2%}")

,quality fact,value
0,Meta-attributed conversation starts,116098
1,Observed Meta-sourced WhatsApp conversations,617
2,Observed / Meta reconciliation ratio,0.005314
3,Organic/direct conversations kept as context,171
4,Open or pending outcomes,61
5,Repeated customers,123
6,Reach greater than impressions rows,118
7,Evidence status,limited_evidence


Coverage = 617 / 116,098 = 0.53%


## 21. Deterministic 70/30 Objective Scenario

The scenario uses 100 learning units:

- **70 exploit units:** only campaigns with a statistically supported SCALE
  decision; allocation is weighted by `probability_better`.
- **30 explore units:** HOLD campaigns become named controlled tests. With no
  business priority input, the MVP divides the reserve equally.
- **KILL:** receives zero.
- Unsupported exploit units remain unallocated rather than being forced into a
  weak campaign.

All three lead campaigns are HOLD, so 70 exploit units remain unallocated and 30
explore units are split into three 10-unit tests.

In [28]:
TOTAL_UNITS = 100.0
EXPLOIT_UNITS = 70.0
EXPLORE_UNITS = 30.0

scale_rows = campaign_scores.loc[campaign_scores["decision"].eq("scale")].copy()
hold_rows = campaign_scores.loc[campaign_scores["decision"].eq("hold")].copy()
allocation_rows = []

if not scale_rows.empty:
    confidence_total = scale_rows["probability_better"].sum()
    for _, row in scale_rows.iterrows():
        units = EXPLOIT_UNITS * row["probability_better"] / confidence_total
        allocation_rows.append({
            "campaign_id": row["campaign_id"], "campaign": row["campaign_name"],
            "pool": "exploit", "decision": "scale", "budget_units": units,
            "reason": "Supported positive lift; weighted by probability better.",
        })

if not hold_rows.empty:
    units = EXPLORE_UNITS / len(hold_rows)
    for _, row in hold_rows.iterrows():
        allocation_rows.append({
            "campaign_id": row["campaign_id"], "campaign": row["campaign_name"],
            "pool": "explore", "decision": "hold", "budget_units": units,
            "reason": "Uncertain lift; fund a named controlled test, not scaling.",
        })

allocations = pd.DataFrame(allocation_rows)
assigned_exploit = allocations.loc[allocations["pool"].eq("exploit"), "budget_units"].sum()
assigned_explore = allocations.loc[allocations["pool"].eq("explore"), "budget_units"].sum()
unallocated = TOTAL_UNITS - assigned_exploit - assigned_explore
display(allocations)
print(f"Exploit assigned = {assigned_exploit:.2f} / {EXPLOIT_UNITS:.2f}")
print(f"Explore assigned = {assigned_explore:.2f} / {EXPLORE_UNITS:.2f}")
print(f"Unallocated = {unallocated:.2f}")
print("Operational = False because outcome reconciliation is limited.")

,campaign_id,campaign,pool,decision,budget_units,reason
0,120209876543220003,Pre-Ramadan Bundle Promo,explore,hold,10.000000,"Uncertain lift; fund a named controlled test, not scaling."
1,120209876543220008,Post-Eid Lookalike Test,explore,hold,10.000000,"Uncertain lift; fund a named controlled test, not scaling."
2,120209876543220010,Lookalike Scale Cycle 3,explore,hold,10.000000,"Uncertain lift; fund a named controlled test, not scaling."


Exploit assigned = 0.00 / 70.00
Explore assigned = 30.00 / 30.00
Unallocated = 70.00
Operational = False because outcome reconciliation is limited.


## 22. Turn Explore Units Into Named Tests

Each HOLD campaign receives a concrete comparison, hypothesis, success rule,
failure rule, and stop rule. The minimum 30 additional mature conversations per
arm is an explicit POC assumption, not a universal marketing benchmark.

The final comparison still uses ranges:

- success: lower 95% direct lift bound above zero;
- failure: upper 95% direct lift bound below zero;
- otherwise: inconclusive and remain HOLD.

In [29]:
test_rows = []
adset_scores = objective_scorecards["adset"]
for _, campaign in hold_rows.iterrows():
    children = adset_scores.loc[
        adset_scores["campaign_id"].eq(campaign["campaign_id"])
    ].sort_values("corrected_score", ascending=False)
    if len(children) >= 2:
        leader = children.iloc[0]
        challenger = children.iloc[1]
        comparison = f"{leader['entity_name']} vs {challenger['entity_name']}"
        hypothesis = (
            f"{leader['entity_name']} will produce a higher mature order creation "
            f"rate than {challenger['entity_name']} under matched conditions."
        )
    else:
        comparison = "Current setup vs one new controlled variant"
        hypothesis = "A single controlled variant will improve mature order creation."
    test_rows.append({
        "campaign_id": campaign["campaign_id"],
        "campaign": campaign["campaign_name"],
        "test": comparison,
        "hypothesis": hypothesis,
        "primary metric": objective_spec["metric"],
        "budget units": EXPLORE_UNITS / len(hold_rows),
        "success rule": "95% lower direct-lift bound > 0 after the stop rule.",
        "failure rule": "95% upper direct-lift bound < 0 after the stop rule.",
        "stop rule": "End of next cycle and at least 30 additional mature conversations per arm; otherwise inconclusive.",
    })
exploration_tests = pd.DataFrame(test_rows)
display(exploration_tests)

,campaign_id,campaign,test,hypothesis,primary metric,budget units,success rule,failure rule,stop rule
0,120209876543220003,Pre-Ramadan Bundle Promo,General audience bundle interested vs 1% Lookalike past bundle buyers,General audience bundle interested will produce a higher mature order creation rate than 1% Lookalike past bundle buyers under matched conditions.,order_creation_rate,10.000000,95% lower direct-lift bound > 0 after the stop rule.,95% upper direct-lift bound < 0 after the stop rule.,End of next cycle and at least 30 additional mature conversations per arm; otherwise inconclusive.
1,120209876543220008,Post-Eid Lookalike Test,5% Lookalike experimental vs 10% Lookalike experimental,5% Lookalike experimental will produce a higher mature order creation rate than 10% Lookalike experimental under matched conditions.,order_creation_rate,10.000000,95% lower direct-lift bound > 0 after the stop rule.,95% upper direct-lift bound < 0 after the stop rule.,End of next cycle and at least 30 additional mature conversations per arm; otherwise inconclusive.
2,120209876543220010,Lookalike Scale Cycle 3,3% Cycle 2 high-LTV scale broader vs 1% Cycle 2 high-LTV scale,3% Cycle 2 high-LTV scale broader will produce a higher mature order creation rate than 1% Cycle 2 high-LTV scale under matched conditions.,order_creation_rate,10.000000,95% lower direct-lift bound > 0 after the stop rule.,95% upper direct-lift bound < 0 after the stop rule.,End of next cycle and at least 30 additional mature conversations per arm; otherwise inconclusive.


## 23. Build The Exact Objective-Only Evidence Payload

The payload contains only the objective contract, deterministic scorecards,
semantic aggregates, data limitations, and the locked budget scenario. Raw
conversation text is not sent to the narrator.

In [30]:
score_fields = [
    "entity_id", "entity_name", "campaign_id", "campaign_name", "objective",
    "mature_orders_created", "mature_conversations", "raw_score",
    "corrected_score", "corrected_score_low", "corrected_score_high",
    "benchmark_score", "benchmark_source", "benchmark_peer_count",
    "expected_lift", "lift_low", "lift_high", "probability_better", "decision",
    "spend", "net_revenue", "net_roas", "observed_conversations",
    "open_or_pending_conversations", "semantic_conversations",
    "high_purchase_intent_conversations", "barrier_conversations",
    "agent_helpful_conversations", "top_conversation_purpose", "top_barrier",
    "top_mentioned_product",
]

def compact_records(frame):
    columns = [column for column in score_fields if column in frame.columns]
    clean = frame[columns].copy().where(pd.notna(frame[columns]), None)
    return clean.to_dict(orient="records")

campaign_payloads = []
for _, campaign in campaign_scores.iterrows():
    campaign_id = campaign["campaign_id"]
    campaign_payloads.append({
        "cycle_id": "sample2-completed-cycle",
        "objective": FOCUS_OBJECTIVE,
        "business_job": objective_spec["business_job"],
        "success_question": objective_spec["success_question"],
        "primary_metric": objective_spec["metric"],
        "campaign": compact_records(campaign_scores.loc[
            campaign_scores["campaign_id"].eq(campaign_id)
        ])[0],
        "adsets": compact_records(objective_scorecards["adset"].loc[
            objective_scorecards["adset"]["campaign_id"].eq(campaign_id)
        ]),
        "ads": compact_records(objective_scorecards["ad"].loc[
            objective_scorecards["ad"]["campaign_id"].eq(campaign_id)
        ]),
        "creatives": compact_records(objective_scorecards["creative"].loc[
            objective_scorecards["creative"]["campaign_id"].eq(campaign_id)
        ]),
        "audiences": compact_records(objective_scorecards["audience"].loc[
            objective_scorecards["audience"]["campaign_id"].eq(campaign_id)
        ]),
        "limitations": quality.warnings + [
            "Semantic schema version 1 is available only for the Post-Eid supplied sample.",
            "Semantic signals are diagnostic and cannot override the outcome decision.",
        ],
    })

post_payload = next(
    item for item in campaign_payloads if item["campaign"]["campaign_name"] == POST_EID_NAME
)
serialized_post_payload = json.dumps(post_payload, sort_keys=True, ensure_ascii=True, default=str)
display(JSON(post_payload, expanded=False))
print(f"Payload characters = {len(serialized_post_payload):,}")
print(f"Payload SHA256 = {hashlib.sha256(serialized_post_payload.encode('utf-8')).hexdigest()}")

<IPython.core.display.JSON object>

Payload characters = 9,974
Payload SHA256 = e7b329dea5bf92ca26df8134461f47548af9f7cc36255fb5016960b3cd0b4340


## 24. Optional Structured LLM Runs

Five calls are available when `RUN_LLM = True`:

1. one campaign analyst call for each of the three lead campaigns;
2. one objective-level synthesis call;
3. one stakeholder recommendation narrator call.

All outputs use strict Pydantic contracts whose fields are required. The default
execution makes no paid API calls. Outputs are cached by model, prompt hash, and
payload hash.

In [31]:
class ObjectiveEvidenceReference(BaseModel):
    entity_level: str
    entity_id: str
    metric: str
    actual: Optional[float]
    benchmark: Optional[float]


class ObjectiveCampaignInsight(BaseModel):
    campaign_id: str
    objective: str
    target_assessment: str
    supporting_evidence: List[ObjectiveEvidenceReference]
    performance_drivers: List[str]
    audience_findings: List[str]
    creative_findings: List[str]
    risks_and_confounders: List[str]
    strategic_lesson: str
    next_controlled_test: Optional[str]
    evidence_status: str


class ObjectivePortfolioInsight(BaseModel):
    cycle_id: str
    objective: str
    repeated_patterns: List[str]
    conflicting_results: List[str]
    strategic_lessons: List[str]
    portfolio_risks: List[str]
    tests_to_prioritize: List[str]


class ObjectiveStakeholderReport(BaseModel):
    cycle_id: str
    objective: str
    executive_summary: str
    business_owner_sections: List[str]
    marketing_director_sections: List[str]
    performance_manager_sections: List[str]
    data_limitations: List[str]


RUN_LLM = False
FORCE_LLM_RERUN = False
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
PROMPT_FILES = {
    "campaign": ROOT / "src_2" / "prompts" / "objective_campaign_analysis.md",
    "synthesis": ROOT / "src_2" / "prompts" / "objective_synthesis.md",
    "narrator": ROOT / "src_2" / "prompts" / "objective_recommendation_narrator.md",
}
prompts = {name: path.read_text(encoding="utf-8") for name, path in PROMPT_FILES.items()}
display(pd.DataFrame([
    {"stage": name, "characters": len(text),
     "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest()}
    for name, text in prompts.items()
]))
print(f"RUN_LLM = {RUN_LLM}; model = {MODEL}")

,stage,characters,sha256
0,campaign,1511,615d65377b10a96b3cbe825315d225411e4c7d9a7d82d04dde044c2cbaf3dd3d
1,synthesis,775,51866a69ce9f1849f5d6fbdddc35c5f1d8023881a55a730b8be7ec34df8f3123
2,narrator,1013,1ba008cd090b9d32a387f31fda6037d43d1ddbd7a154c843ffe5209372386a32


RUN_LLM = False; model = gpt-5-mini


## 25. Run, Validate, And Cache Campaign Analysis

Pydantic validates structure. The grounding audit in the next cell validates that
every numeric evidence reference maps back to the deterministic payload.

In [32]:
def stable_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=True, default=str)


def cached_parse(stage, prompt_key, payload, output_model):
    from openai import OpenAI

    prompt_sha = hashlib.sha256(prompts[prompt_key].encode("utf-8")).hexdigest()
    input_sha = hashlib.sha256(stable_json(payload).encode("utf-8")).hexdigest()
    cache_key = hashlib.sha256(
        f"{MODEL}|{prompt_sha}|{input_sha}".encode("utf-8")
    ).hexdigest()[:20]
    path = LLM_CACHE / f"{stage}_{cache_key}.json"
    if path.exists() and not FORCE_LLM_RERUN:
        saved = json.loads(path.read_text(encoding="utf-8"))
        return output_model.model_validate(saved["output"]), "cache", path

    client = OpenAI()
    response = client.responses.parse(
        model=MODEL,
        instructions=prompts[prompt_key],
        input=stable_json(payload),
        text_format=output_model,
    )
    if response.output_parsed is None:
        raise RuntimeError("The model did not return a validated structured response")
    result = response.output_parsed
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(stable_json({
        "model": MODEL, "prompt_sha256": prompt_sha,
        "input_sha256": input_sha, "output": result.model_dump(mode="json"),
    }), encoding="utf-8")
    return result, "OpenAI", path


campaign_insights = []
llm_audit = []
if RUN_LLM:
    for payload in campaign_payloads:
        result, source, path = cached_parse(
            f"campaign_{payload['campaign']['campaign_id']}",
            "campaign", payload, ObjectiveCampaignInsight,
        )
        campaign_insights.append(result)
        llm_audit.append({
            "campaign": payload["campaign"]["campaign_name"],
            "source": source, "cache": str(path.relative_to(ROOT)),
        })
    display(pd.DataFrame(llm_audit))
    display(JSON([item.model_dump(mode="json") for item in campaign_insights], expanded=True))
else:
    print("Campaign LLM calls skipped. Set RUN_LLM = True and rerun from this cell.")

Campaign LLM calls skipped. Set RUN_LLM = True and rerun from this cell.


## 26. Grounding Audit

Schema validation proves the output shape. Grounding validation proves that a
cited entity, metric, actual value, and benchmark exist in the locked payload.

In [33]:
payload_by_campaign = {
    item["campaign"]["campaign_id"]: item for item in campaign_payloads
}

def evidence_lookup(payload):
    lookup = {}
    for level_key, level_name in (
        ("campaign", "campaign"), ("adsets", "adset"), ("ads", "ad"),
        ("creatives", "creative"), ("audiences", "audience"),
    ):
        entities = [payload[level_key]] if level_key == "campaign" else payload[level_key]
        for entity in entities:
            for metric in (
                "raw_score", "corrected_score", "expected_lift", "lift_low",
                "lift_high", "probability_better", "spend", "net_revenue", "net_roas",
            ):
                if metric in entity:
                    benchmark = entity.get("benchmark_score") if metric in {
                        "raw_score", "corrected_score"
                    } else (0.0 if metric in {"expected_lift", "lift_low", "lift_high"} else None)
                    lookup[(level_name, entity["entity_id"], metric)] = (
                        entity.get(metric), benchmark
                    )
    return lookup

grounding_rows = []
for insight in campaign_insights:
    payload = payload_by_campaign.get(insight.campaign_id)
    lookup = evidence_lookup(payload) if payload else {}
    for reference in insight.supporting_evidence:
        key = (reference.entity_level, reference.entity_id, reference.metric)
        expected = lookup.get(key)
        actual_ok = expected is not None and (
            expected[0] is None and reference.actual is None
            or expected[0] is not None and reference.actual is not None
            and abs(expected[0] - reference.actual) < 1e-9
        )
        benchmark_ok = expected is not None and (
            reference.benchmark is None
            or expected[1] is not None
            and abs(expected[1] - reference.benchmark) < 1e-9
        )
        grounding_rows.append({
            "campaign_id": insight.campaign_id, "reference": key,
            "exists": expected is not None, "actual matches": actual_ok,
            "benchmark matches": benchmark_ok,
        })

grounding_audit = pd.DataFrame(grounding_rows)
if grounding_audit.empty:
    print("Grounding audit will run after campaign LLM calls.")
else:
    display(grounding_audit)
    assert grounding_audit["exists"].all()
    assert grounding_audit["actual matches"].all()
    assert grounding_audit["benchmark matches"].all()

Grounding audit will run after campaign LLM calls.


## 27. Objective Synthesis And Recommendation Narration

The synthesis sees only deterministic decisions plus validated campaign insights.
The narrator then sees the synthesis and the already-calculated budget scenario.
Neither stage can create or alter budget numbers.

In [34]:
objective_insight = None
stakeholder_report = None
if RUN_LLM:
    synthesis_payload = {
        "cycle_id": "sample2-completed-cycle",
        "objective": FOCUS_OBJECTIVE,
        "deterministic_campaign_results": compact_records(campaign_scores),
        "campaign_insights": [item.model_dump(mode="json") for item in campaign_insights],
    }
    objective_insight, synthesis_source, _ = cached_parse(
        "objective_synthesis", "synthesis", synthesis_payload,
        ObjectivePortfolioInsight,
    )
    narrator_payload = {
        "cycle_id": "sample2-completed-cycle",
        "objective": FOCUS_OBJECTIVE,
        "objective_insight": objective_insight.model_dump(mode="json"),
        "budget_scenario": {
            "total_units": TOTAL_UNITS,
            "exploit_units": EXPLOIT_UNITS,
            "explore_units": EXPLORE_UNITS,
            "allocations": allocations.to_dict(orient="records"),
            "tests": exploration_tests.to_dict(orient="records"),
            "unallocated_units": unallocated,
            "operational": False,
        },
    }
    stakeholder_report, narrator_source, _ = cached_parse(
        "objective_report", "narrator", narrator_payload,
        ObjectiveStakeholderReport,
    )
    print(f"Synthesis source = {synthesis_source}; narrator source = {narrator_source}")
    display(JSON(objective_insight.model_dump(mode="json"), expanded=True))
    display(JSON(stakeholder_report.model_dump(mode="json"), expanded=True))
else:
    print("Synthesis and recommendation narration skipped until RUN_LLM = True.")

Synthesis and recommendation narration skipped until RUN_LLM = True.


## 28. Final Post-Eid Decision Card

This table remains available offline. The optional LLM can explain it, but cannot
replace it.

In [35]:
final_post_eid = pd.DataFrame([{
    "campaign": POST_EID_NAME,
    "objective": FOCUS_OBJECTIVE,
    "raw evidence": f"{successes} created orders / {mature} mature conversations",
    "unresolved excluded": f"{unresolved} / {observed}",
    "raw score": post_row["raw_score"],
    "corrected score": post_row["corrected_score"],
    "corrected range": f"{post_row['corrected_score_low']:.2%} to {post_row['corrected_score_high']:.2%}",
    "same-objective benchmark": post_row["benchmark_score"],
    "favorable lift range": f"{post_row['lift_low']:+.2%} to {post_row['lift_high']:+.2%}",
    "probability better": post_row["probability_better"],
    "decision": post_row["decision"].upper(),
    "next action": "Run the named 5% versus 10% lookalike test; do not scale yet.",
    "semantic evidence": "41/63 high intent; 30/63 had barriers; schema v1 only.",
    "data status": "Illustrative, non-operational until population reconciliation is understood.",
}])
display(final_post_eid.style.format({
    "raw score": "{:.2%}", "corrected score": "{:.2%}",
    "same-objective benchmark": "{:.2%}", "probability better": "{:.2%}",
}))

if stakeholder_report is not None:
    display(Markdown("### LLM Executive Summary"))
    display(Markdown(stakeholder_report.executive_summary))

,campaign,objective,raw evidence,unresolved excluded,raw score,corrected score,corrected range,same-objective benchmark,favorable lift range,probability better,decision,next action,semantic evidence,data status
0,Post-Eid Lookalike Test,OUTCOME_LEADS,38 created orders / 52 mature conversations,11 / 63,73.08%,73.90%,64.66% to 82.35%,75.00%,-16.22% to +16.19%,44.04%,HOLD,Run the named 5% versus 10% lookalike test; do not scale yet.,41/63 high intent; 30/63 had barriers; schema v1 only.,"Illustrative, non-operational until population reconciliation is understood."


## 29. Calculation And Responsibility Ledger

| Stage | Evidence | Post-Eid output | Why it matters | Owner |
|---|---|---|---|---|
| Eligibility | 63 observed, 11 unresolved | 52 mature | Avoids false failures | Deterministic |
| Raw score | 38 / 52 | 73.08% | Describes observed sample | Deterministic |
| Raw interval | 38 successes, 52 trials | 59.75% to 83.23% | Shows raw uncertainty | Deterministic |
| Peer evidence | 58 / 77 from two other lead campaigns | 75.32% raw | Adds objective-specific context | Deterministic |
| Half-count adjustment | +0.5 success, +0.5 failure | 75.00% | Avoids degenerate 0% or 100% prior | Deterministic |
| Empirical prior | peer center and conservative strength | alpha 28.875, beta 9.625 | Controls how much peers contribute | Deterministic |
| Corrected score | prior + Post-Eid counts | 73.90% | Reduces small-sample overreaction | Deterministic |
| Corrected range | posterior simulation | about 64.6% to 82.2% | Shows plausible campaign rate | Deterministic |
| Favorable lift | campaign draws minus peer draws | range crosses zero | Prevents a noisy winner call | Deterministic |
| Decision | full lift range | HOLD | Connects uncertainty to action | Deterministic |
| Conversation semantics | 63 schema-v1 records | intent, barriers, products, helpfulness | Suggests why and what to test | LLM extraction + deterministic aggregation |
| Budget | locked decisions and 70/30 policy | 70 unallocated, 30 explore | Avoids forced scaling | Deterministic |
| Campaign explanation | exact evidence payload | structured insight | Makes evidence understandable | Optional LLM |
| Objective synthesis | validated campaign insights | patterns and tests | Builds strategy across lead campaigns | Optional LLM |
| Stakeholder report | synthesis + locked budget | role-specific narrative | Communicates without changing numbers | Optional LLM |

## Final Reading Rule

Do not start with the largest raw rate. Start with eligible evidence, then read the
corrected score, benchmark, lift range, and range-based decision. Use conversation
semantics to form a hypothesis and design the next test. Use the LLM only to
explain the locked result.